# Treasury Bond Futures CTD & Delivery Option Calculator

This Colab notebook estimates the **quality (bond-selection) delivery option** in a Treasury bond futures contract. It follows the mechanism in *A Fat Cat in a Small Door* (Convexity Maven, 13 Oct 2026): the short futures position can choose an eligible bond, and the cheapest-to-deliver (CTD) bond can change as yields move.

The notebook calculates bond prices and conversion-factor-adjusted delivery costs, identifies today’s CTD, measures CTD switching over yield scenarios, and estimates the quality option value with a two-factor Monte Carlo model.

**Model scope:** This is a transparent scenario model, not an exchange-grade futures valuation. The option estimate is only as good as the basket, yields, volatility assumptions, and delivery date entered below. It models the quality option only; it does not model the wild-card, timing, end-of-month, repo specialness, delivery-month carry, cheapest financing, or delivery constraints. The source PDF supplies examples and selected conversion factors, but no volatility surface or full current basket.


## 1. Model and interpretation

At delivery, each eligible bond has a modeled dirty price `P_i`. Its conversion-adjusted delivery cost is `P_i / CF_i`, where `CF_i` is the exchange conversion factor. The short selects the bond with the lowest adjusted cost. The current CTD is the bond with the lowest adjusted cost using current yields.

For a simple quality-option estimate, the current CTD is used as the reference bond. In each simulated delivery scenario, the quality-option payoff per $100 par is:

`max(reference bond dirty price / reference CF - minimum(delivery bond dirty price / delivery CF), 0)`

The mean payoff is discounted to today. Scenarios use correlated normal **yield level** and **curve slope** changes; bond prices are fully repriced at the simulated yields. This is a zero-drift, lognormal-free approximation under a risk-neutral-style assumption. It is useful for sensitivity and relative-risk work, but it is not a calibrated arbitrage-free Hull-White price. Prefer market-implied factor volatilities where available; do not interpret historical realized volatility as a risk-neutral input without adjustment.

The input basket below is a small illustrative subset of the highlighted rows in the PDF, not the full 59-bond December 2026 basket and not a current market data feed. Replace it with the contract’s full eligible basket, current conversion factors, and current yields/prices before relying on the outputs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, datetime
from IPython.display import display

np.random.seed(42)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Contract and market inputs

Edit the dates and assumptions. `level_vol_bp` is the standard deviation of the parallel yield move over the current-to-delivery horizon. `slope_vol_bp` is the standard deviation of a 10s/30s-style slope move over that same horizon. For each bond, the slope factor changes its yield by `(maturity_years - 20) / 10 × slope_move`; this is a simple, explicit proxy. Adjust the factor definition or bond-specific factor loadings when you have a calibrated curve model.


In [ ]:
# Contract / valuation inputs
valuation_date = date(2026, 10, 13)
delivery_date = date(2026, 12, 1)  # replace with the contract's actual delivery window/date
futures_price = 102.25             # informational; used for invoice/net-basis diagnostics
risk_free_discount_rate_pct = 4.0  # continuously compounded discount rate for option PV

# Yield-factor volatility assumptions over the full horizon (basis points)
level_vol_bp = 45.0
slope_vol_bp = 20.0
level_slope_correlation = -0.20
number_of_scenarios = 50000

# Illustrative highlighted bonds from the PDF's December 2026 conversion table.
# Coupon is percent per year; yield_pct is a user-editable illustrative yield.
# Replace / extend with the full live eligible basket and observed yields.
basket = pd.DataFrame([
    {"cusip":"912810SS8", "coupon_pct":1.625, "maturity":"2050-11-15", "conversion_factor":0.4499, "yield_pct":5.68},
    {"cusip":"912810SU3", "coupon_pct":1.875, "maturity":"2051-02-15", "conversion_factor":0.4789, "yield_pct":5.68},
    {"cusip":"912810RK6", "coupon_pct":2.500, "maturity":"2045-02-15", "conversion_factor":0.6179, "yield_pct":5.68},
    {"cusip":"912810TW8", "coupon_pct":4.750, "maturity":"2043-11-15", "conversion_factor":0.8690, "yield_pct":5.68},
    {"cusip":"912810U55", "coupon_pct":4.750, "maturity":"2045-02-15", "conversion_factor":0.8635, "yield_pct":5.68},
])

display(basket)


## 3. Bond pricing and current CTD

Prices are computed per $100 par from semiannual coupon cash flows using nominal annual yield compounded semiannually. Dates use an approximate actual-days coupon fraction. The model reports clean and dirty prices; delivery cost uses the dirty price because accrued interest is included in the invoice settlement.


In [ ]:
def coupon_dates_backwards(maturity, settlement, frequency=2):
    """Approximate regular coupon schedule by stepping back 6 calendar months."""
    from dateutil.relativedelta import relativedelta
    step = relativedelta(months=12 // frequency)
    dates = [maturity]
    while dates[-1] > settlement:
        dates.append(dates[-1] - step)
    return list(reversed(dates))  # includes last coupon on/before settlement and maturity

def bond_price(coupon_pct, maturity, settlement, yield_pct, face=100.0):
    maturity = pd.Timestamp(maturity).date()
    settlement = pd.Timestamp(settlement).date()
    if maturity <= settlement:
        raise ValueError("Bond maturity must be after settlement")
    dates = coupon_dates_backwards(maturity, settlement)
    prev_coupon, next_coupon = dates[0], dates[1]
    period_days = (next_coupon - prev_coupon).days
    elapsed = (settlement - prev_coupon).days
    accrued = face * coupon_pct / 100 / 2 * elapsed / period_days
    first_fraction = (next_coupon - settlement).days / period_days
    future_dates = dates[1:]
    y = yield_pct / 100
    if 1 + y / 2 <= 0:
        raise ValueError("Yield is outside the supported range")
    dirty = 0.0
    coupon = face * coupon_pct / 100 / 2
    for j, paydate in enumerate(future_dates):
        periods = first_fraction + j
        cashflow = coupon + (face if paydate == maturity else 0.0)
        dirty += cashflow / (1 + y / 2) ** (2 * periods)
    return dirty - accrued, dirty, accrued

def add_current_prices(frame, settlement):
    out = frame.copy()
    prices = [bond_price(r.coupon_pct, r.maturity, settlement, r.yield_pct)
              for r in out.itertuples()]
    out[["clean_price", "dirty_price", "accrued"]] = pd.DataFrame(prices, index=out.index)
    out["adjusted_delivery_cost"] = out["dirty_price"] / out["conversion_factor"]
    out["implied_futures_price"] = out["clean_price"] / out["conversion_factor"]
    out["net_basis_points"] = (out["clean_price"] - futures_price * out["conversion_factor"]) * 100
    return out.sort_values("adjusted_delivery_cost").reset_index(drop=True)

current = add_current_prices(basket, valuation_date)
ctd = current.iloc[0]
print(f"Current modeled CTD: {ctd.cusip}")
print(f"Current CTD adjusted dirty delivery cost: {ctd.adjusted_delivery_cost:.4f} per $100 par")
display(current[["cusip", "coupon_pct", "maturity", "conversion_factor", "yield_pct", "clean_price", "dirty_price", "adjusted_delivery_cost", "net_basis_points"]])


## 4. Yield scenarios and delivery-option estimate

The simulation applies correlated level and slope changes to the entered yields. The CTD is recomputed after repricing every bond at the delivery date. The reported Monte Carlo standard error is sampling error only; it does not capture model or input uncertainty.


In [ ]:
def simulate_quality_option(frame, valuation_date, delivery_date, reference_cusip,
                             level_vol_bp, slope_vol_bp, correlation,
                             discount_rate_pct, n_scenarios=50000, seed=42):
    if not (-1 <= correlation <= 1):
        raise ValueError("Correlation must be between -1 and 1")
    horizon = (pd.Timestamp(delivery_date) - pd.Timestamp(valuation_date)).days / 365.0
    if horizon <= 0:
        raise ValueError("Delivery date must be after valuation date")
    rng = np.random.default_rng(seed)
    z = rng.multivariate_normal([0, 0], [[1, correlation], [correlation, 1]], size=n_scenarios)
    level = z[:, 0] * level_vol_bp / 10000
    slope = z[:, 1] * slope_vol_bp / 10000
    maturities = (pd.to_datetime(frame["maturity"]) - pd.Timestamp(delivery_date)).dt.days.to_numpy() / 365.25
    base_yields = frame["yield_pct"].to_numpy() / 100
    cfs = frame["conversion_factor"].to_numpy()
    coupons = frame["coupon_pct"].to_numpy()
    ref_idx = int(frame.index[frame["cusip"] == reference_cusip][0])
    adjusted = np.empty((n_scenarios, len(frame)))
    for s in range(n_scenarios):
        for i in range(len(frame)):
            scenario_yield_pct = (base_yields[i] + level[s] + slope[s] * (maturities[i] - 20) / 10) * 100
            _, dirty, _ = bond_price(coupons[i], frame.iloc[i]["maturity"], delivery_date, scenario_yield_pct)
            adjusted[s, i] = dirty / cfs[i]
    cheapest_idx = adjusted.argmin(axis=1)
    reference_cost = adjusted[:, ref_idx]
    cheapest_cost = adjusted[np.arange(n_scenarios), cheapest_idx]
    payoff = np.maximum(reference_cost - cheapest_cost, 0.0)
    discount = np.exp(-discount_rate_pct / 100 * horizon)
    pv = payoff * discount
    switch_prob = np.mean(cheapest_idx != ref_idx)
    summary = {
        "horizon_years": horizon,
        "reference_cusip": reference_cusip,
        "option_pv_per_100_par": pv.mean(),
        "option_pv_per_1000_par": pv.mean() * 10,
        "undiscounted_expected_payoff": payoff.mean(),
        "mc_standard_error": pv.std(ddof=1) / np.sqrt(n_scenarios),
        "probability_current_ctd_is_not_cheapest": switch_prob,
        "probability_option_payoff_is_positive": np.mean(payoff > 0),
        "discount_factor": discount,
    }
    return summary, adjusted, cheapest_idx, payoff

summary, scenario_costs, delivery_ctd_idx, option_payoffs = simulate_quality_option(
    basket.reset_index(drop=True), valuation_date, delivery_date, ctd.cusip,
    level_vol_bp, slope_vol_bp, level_slope_correlation,
    risk_free_discount_rate_pct, number_of_scenarios
)
display(pd.Series(summary, name="Model output").to_frame())


## 5. CTD-switch and payoff diagnostics


In [ ]:
ctd_prob = pd.Series(delivery_ctd_idx).value_counts(normalize=True).reindex(range(len(basket)), fill_value=0)
ctd_table = pd.DataFrame({"cusip": basket.reset_index(drop=True)["cusip"],
                          "simulated_delivery_ctd_probability": ctd_prob.values})
display(ctd_table.sort_values("simulated_delivery_ctd_probability", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(ctd_table["cusip"], ctd_table["simulated_delivery_ctd_probability"] * 100)
axes[0].set_ylabel("Probability (%)")
axes[0].set_title("Cheapest-to-deliver at delivery")
axes[0].tick_params(axis="x", rotation=45)
axes[1].hist(option_payoffs, bins=50, color="#245b8f", alpha=0.85)
axes[1].set_xlabel("Quality-option payoff per $100 par")
axes[1].set_ylabel("Scenarios")
axes[1].set_title("Undiscounted payoff distribution")
plt.tight_layout()
plt.show()


## 6. Sensitivity to yield volatility

The table below scales both factor volatilities together. This is an input sensitivity, not a forecast. A richer implementation would calibrate a multi-factor term-structure model to swaption or Treasury option prices and include rate drift, curve key-rate factors, repo financing, coupon carry, and the actual delivery window.


In [ ]:
vol_rows = []
for scale in [0.50, 0.75, 1.00, 1.25, 1.50]:
    s, _, _, _ = simulate_quality_option(
        basket.reset_index(drop=True), valuation_date, delivery_date, ctd.cusip,
        level_vol_bp * scale, slope_vol_bp * scale, level_slope_correlation,
        risk_free_discount_rate_pct, max(5000, number_of_scenarios // 5), seed=123
    )
    vol_rows.append({"volatility_scale": scale,
                     "level_vol_bp": level_vol_bp * scale,
                     "slope_vol_bp": slope_vol_bp * scale,
                     "option_pv_per_100_par": s["option_pv_per_100_par"],
                     "ctd_switch_probability": s["probability_current_ctd_is_not_cheapest"]})
display(pd.DataFrame(vol_rows))


## 7. Practical use and source notes

1. Replace the illustrative subset with every bond eligible for the exact contract and delivery month. Validate identifiers, coupons, maturity dates, and conversion factors against the exchange.
2. Enter observed yields or prices from a consistent timestamp and settlement convention. Check the model’s CTD against the market net-basis / implied-repo screen.
3. Supply calibrated risk-neutral level and slope volatilities for the valuation horizon. Run the volatility sensitivity and compare the modeled premium with market implied basis behavior.
4. Add the omitted delivery, financing, and carry options before using this as a fair-value model.

The source commentary says the December 2026 basket has 59 eligible issues and highlights example CTD duration changes: 11.03, 11.63, 13.24, and above 17 as the 20-year yield approaches or exceeds 6%. It motivates a delivery-option model but does not establish a calibrated option premium. See *Convexity Maven, “A Fat Cat in a Small Door,” pages 2–6 (13 October 2026)*. The selected example conversion factors in the input table are read from the page 3 conversion-factor table.

This notebook is for research and educational use, not investment advice. Results are model outputs and can be materially wrong if the input basket, conversion factors, yields, or volatility assumptions are stale or incomplete.
